# Flow-v2 Prompt 3R — L4 smoke and short Fold-0 pilot

This Run-All notebook is the only GPU runner for Prompt 3R. It fails closed at provenance, tests, smoke, and paired-safety boundaries. It does not launch the deferred ablation grid.

In [ ]:
from pathlib import Path
COLAB_ROOT = Path('/content')
DRIVE_ROOT = Path('/content/drive/MyDrive/ToothFairy/ToothFairy3/iac_runs')
REPO_DIR = COLAB_ROOT / 'ToothFairy3-IAC-Segmentation-Flow'
REPO_URL = 'https://github.com/ColdVI/ToothFairy3-IAC-Segmentation-Flow.git'
PINNED_COMMIT = '443ae1eac2771a43a71a9b4f1093082b4fe66cb8'
DATASET_ROOT = DRIVE_ROOT / 'dataset_cache_colab_v1/Dataset801_IAC_LR'
CACHE_ROOT = DRIVE_ROOT / 'sdf_cache_backup'
IMAGES_DIR = DATASET_ROOT / 'imagesTr'
LABELS_DIR = DATASET_ROOT / 'labelsTr'
GT_SDF_DIR = CACHE_ROOT / 'gt_sdf'
COARSE_SDF_DIR = CACHE_ROOT / 'coarse_sdf'
DRIVE_SPLITS = DRIVE_ROOT / 'configs_cache/splits.json'
IDENTITY_JSON = DRIVE_ROOT / 'outputs/baselines/identity_prior.json'
PROMPT2_DIR = DRIVE_ROOT / 'outputs/analysis'
PROMPT3R_ROOT = DRIVE_ROOT / 'outputs/prompt3r'
SMOKE_ROOT = PROMPT3R_ROOT / 'smoke'
REQUIRE_GPU_NAME = 'L4'
MIN_FREE_GB = 10


In [ ]:
from google.colab import drive
import os, subprocess, sys
drive.mount(str(COLAB_ROOT / 'drive'))
if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'fetch', '--all', '--tags', '--prune'], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', '--detach', PINNED_COMMIT], cwd=REPO_DIR, check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True).strip()
assert head == PINNED_COMMIT, (head, PINNED_COMMIT)
assert not dirty, f'Dirty pinned checkout: {dirty}'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nibabel>=5', 'scipy>=1.10', 'scikit-image>=0.21', 'pyyaml>=6', 'matplotlib', 'pytest'], check=True)
os.chdir(REPO_DIR)
print({'pinned_commit': head, 'repo': str(REPO_DIR)})


In [ ]:
import hashlib, json, shutil, torch, numpy as np
def file_sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
repo_splits = REPO_DIR / 'configs/splits.json'
assert DRIVE_SPLITS.is_file(), DRIVE_SPLITS
assert file_sha(repo_splits) == file_sha(DRIVE_SPLITS), 'Repo/Drive split drift'
PROMPT3R_ROOT.mkdir(parents=True, exist_ok=True)
preflight_receipt = PROMPT3R_ROOT / 'preflight_receipt.json'
preflight_cmd = [sys.executable, 'scripts/prompt3r_preflight.py',
    '--drive-root', str(DRIVE_ROOT), '--images', str(IMAGES_DIR),
    '--labels', str(LABELS_DIR), '--gt-sdf', str(GT_SDF_DIR),
    '--coarse-sdf', str(COARSE_SDF_DIR), '--identity-json', str(IDENTITY_JSON),
    '--prompt2-dir', str(PROMPT2_DIR), '--output-root', str(PROMPT3R_ROOT),
    '--out', str(preflight_receipt), '--require-gpu-name', REQUIRE_GPU_NAME,
    '--min-free-gb', str(MIN_FREE_GB)]
subprocess.run(preflight_cmd, cwd=REPO_DIR, check=True)
PREFLIGHT = json.loads(preflight_receipt.read_text())
assert PREFLIGHT['ready'] and REQUIRE_GPU_NAME.lower() in PREFLIGHT['gpu'].lower()
PREFLIGHT_PASSED = True
print({'gpu': torch.cuda.get_device_name(0), 'cuda': torch.version.cuda, 'free_gb': PREFLIGHT['free_gb']})


In [ ]:
assert PREFLIGHT_PASSED
test_result = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/'], cwd=REPO_DIR)
assert test_result.returncode == 0, 'Test gate failed; smoke and pilot are forbidden'
TEST_GATE_PASSED = True
print('TEST GATE PASSED')


In [ ]:
assert PREFLIGHT_PASSED and TEST_GATE_PASSED
common = ['--config', str(REPO_DIR/'configs/flow_prompt3r.yaml'),
    '--identity-json', str(IDENTITY_JSON), '--splits', str(repo_splits),
    '--panel-config', str(REPO_DIR/'configs/prompt3r_pilot_cases.json'),
    '--fold', '0', '--images', str(IMAGES_DIR), '--labels', str(LABELS_DIR),
    '--gt-sdf', str(GT_SDF_DIR), '--coarse-sdf', str(COARSE_SDF_DIR), '--device', 'cuda']
smoke_last = SMOKE_ROOT / 'last.pt'
smoke_epoch = torch.load(smoke_last, map_location='cpu', weights_only=False)['epoch'] if smoke_last.is_file() else -1
if smoke_epoch < 1:
    command = [sys.executable, 'flow/prompt3r_train.py', *common, '--out', str(SMOKE_ROOT), '--smoke', '--stop-after-epoch', '1']
    if (SMOKE_ROOT / 'manifest.json').is_file(): command.append('--resume')
    subprocess.run(command, cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, 'flow/prompt3r_train.py', *common, '--out', str(SMOKE_ROOT), '--smoke', '--resume'], cwd=REPO_DIR, check=True)
smoke_trajectory = json.loads((SMOKE_ROOT/'epoch_trajectory.json').read_text())
assert [row['epoch'] for row in smoke_trajectory] == [0, 1, 2]
assert all(row['total'] is None or np.isfinite(row['total']) for row in smoke_trajectory)
assert all((SMOKE_ROOT/name).is_file() for name in ['epoch_000_untrained.pt', 'epoch_001.pt', 'epoch_002.pt', 'pilot_decision.json'])
SMOKE_PASSED = True
print('SMOKE GATE PASSED: interruption/resume, finite loss, immutable epochs')


In [ ]:
assert PREFLIGHT_PASSED and TEST_GATE_PASSED and SMOKE_PASSED
pilot_cmd = [sys.executable, 'flow/prompt3r_train.py', *common, '--out', str(PROMPT3R_ROOT)]
if (PROMPT3R_ROOT / 'manifest.json').is_file(): pilot_cmd.append('--resume')
subprocess.run(pilot_cmd, cwd=REPO_DIR, check=True)
PILOT_FINISHED = True


In [ ]:
assert PILOT_FINISHED
required = ['pilot_case_ids.json', 'resolved_config.yaml', 'progress.csv',
    'epoch_trajectory.csv', 'epoch_trajectory.json',
    'paired_validation_quick_all_epochs.csv', 'paired_validation_full.csv',
    'pilot_report.md', 'pilot_decision.json', 'trajectory_metrics.pdf',
    'geometry_bias.pdf', 'manifest.json']
missing = [name for name in required if not (PROMPT3R_ROOT/name).is_file() or (PROMPT3R_ROOT/name).stat().st_size == 0]
assert not missing, missing
trajectory = json.loads((PROMPT3R_ROOT/'epoch_trajectory.json').read_text())
assert [row['epoch'] for row in trajectory] == list(range(16))
assert len({row['epoch'] for row in trajectory}) == 16
assert (PROMPT3R_ROOT/'epoch_000_untrained.pt').is_file()
assert all((PROMPT3R_ROOT/f'epoch_{epoch:03d}.pt').is_file() for epoch in range(1, 16))
decision = json.loads((PROMPT3R_ROOT/'pilot_decision.json').read_text())
assert decision['decision'] in {'PROMOTE_TO_REVISED_STAGE1_ABLATION', 'STOP_FLOW_V2_AND_REOPEN_DIAGNOSIS'}
manifest = json.loads((PROMPT3R_ROOT/'manifest.json').read_text())
assert manifest['status'] == 'completed' and manifest['selection_policy'] == 'paired_identity'
print(json.dumps({'decision': decision, 'best_safe': manifest['metrics']['best_safe'], 'output_root': str(PROMPT3R_ROOT)}, indent=2))
